# Scale across UrGs (Lee)

Merged entry for the four overpass times. Paired filename variant of `03-UrG Scale Effect.ipynb`: same corrected shared methods under `00-Support/lstnat` and the same common 61-scale tables written by `02-Scale Stats`; stored outputs are cleared.


In [ ]:
from pathlib import Path
import subprocess
import sys
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "00-Support" / "lstnat").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside Upload Code")
sys.path.insert(0, str(ROOT / "00-Support"))
from lstnat.settings import TIMES, load, output_path
CONFIG = ROOT / "00-Support" / "config" / "local.json"
if not CONFIG.exists():
    CONFIG = ROOT / "00-Support" / "config" / "example.json"
CFG = load(CONFIG)
def run_command(*args):
    return subprocess.run([sys.executable, str(ROOT / "00-Support" / "run.py"), *args,
                           "--config", str(CONFIG)], cwd=ROOT, check=True)
def run_script(relative):
    return subprocess.run([sys.executable, str(ROOT / relative), "--config", str(CONFIG)],
                          cwd=ROOT, check=True)


Run `02-Scale Stats` first. Each pass below reads one overpass time from the common tables. Zone values are local contributions from the global spatial graph; no manually ranked knees or offsets are used. This file is the paired `(Lee)` variant of the same corrected analysis.


In [ ]:
USE_LEGACY = False


In [ ]:
import pandas as pd

FOLDER = "legacy_diagnostic" if USE_LEGACY else "scale"
curves = pd.read_csv(output_path(CFG, FOLDER, "curves.csv"), keep_default_na=False, na_values=[""])
knees = pd.read_csv(output_path(CFG, FOLDER, "knees.csv"), keep_default_na=False, na_values=[""])

for time, spec in TIMES.items():
    zone_curves = curves[(curves.time == time) & (curves.metric == "mean_local_lee")]
    zone_knees = knees[(knees.time == time) & (knees.metric == "mean_local_lee")]
    print(f"{time} - {spec['name']} ({spec['nat']})")
    display(zone_curves.pivot(index="scale_m", columns="zone", values="value"))
    display(zone_knees)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (time, spec) in zip(axes.flat, TIMES.items()):
    subset = curves[(curves.time == time) & (curves.metric == "mean_local_lee")]
    for zone, group in subset.groupby("zone"):
        ax.plot(group.scale_m, group.smoothed, label=zone)
    ax.set(title=f"{time} - {spec['name']}", xlabel="Aggregation scale (m)",
           ylabel="Mean local Lee L")
axes.flat[0].legend(frameon=False)
fig.tight_layout()
plt.show()


Knee summary: selected scale (m) for every metric, zone, and overpass time.


In [ ]:
display(knees.pivot_table(index=["metric", "zone"], columns="time", values="scale", aggfunc="first"))
